<a href="https://colab.research.google.com/github/swartzbt/InverseDesignWorkshop_AISS26/blob/main/InverseDesignHologram.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Inverse Design with PyTorch

**Duration:** 45 minutes

Design a simple diffractive phase hologram using PyTorch's automatic differentiation.

## 1. Environment Setup

In this workshop we'll use PyTorch as an engineering design tool.

**Objectives**
- Use autograd
- Build a custom `torch.nn.Module`
- Define a custom merit function
- Optimize a physical system

Run the following cell to import dependencies.

In [ ]:
!wget -q https://raw.githubusercontent.com/swartzbt/InverseDesignWorkshop_AISS26/main/goat.jpg

import torch
import torch.nn as nn
import torch.fft as fft
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image, ImageOps
from google.colab import files

torch.manual_seed(0)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

## 2. Autograd

We'll verify autograd on a simple differentiable function.

For f(x)=(x-1.5)^2, compare the autograd derivative with the analytical result.

In [ ]:
x = torch.tensor(2.0, requires_grad=True)  # Variables with requires_grad=True will have their gradients tracked
f = lambda x: (x - 1.5) ** 2  # Define a forward calculation
y = f(x)
y.backward()                 # Running y.backward() will calculate the gradients of y with respect to each tracked input
print("Autograd:",x.grad.item())

# Compare with analytical result
f_prime = lambda x: 2 * (x - 1.5)
print("Analytical:", f_prime(x).item())


## 3. Gradient Descent

Use gradient descent to find the minimum of the function f(x)=(x-1.5)^2 defined in the previous section.

In [ ]:
x = torch.tensor(5.0, requires_grad=True)

lr = 0.3
for i in range(10):
    y = f(x)
    y.backward()
    with torch.no_grad():
        x -= lr*x.grad
    x.grad.zero_()
    print(f'Iter {i+1:2d}: x = {x.item():.4f}')


Pretty simple, right?

This basic framework is used for all gradient descent in PyTorch, whether we are training an AI model or engineering an optimized system.

We define a model we want to train, then a loss (or merit) function that quantifies how good the model is doing, and finally use gradient descent to adjust the parameters of the model to optimize the performance.

## 4. Custom Module

Now we model a hologram.

The trainable parameter is a phase matrix. The forward model computes a Fraunhofer diffraction pattern using an FFT.

**Instructor note:** In the live workshop we'll discuss why an FFT models far-field diffraction.

In [ ]:
def zeropad(original, n):
    """ Increases the size of a tensor by zero padding last 2 dimensions. """
    assert original.size(-1) == original.size(-2)
    n_in = original.size(-1)
    pre = (n + n_in%2 - n_in) // 2
    post = (n + n%2 - n_in) // 2
    return F.pad(original, (pre, post, pre, post), 'constant', 0)

def unpad(original, n):
    """ Returns a view of the center (n x n) region of a torch.tensor. """
    assert original.size(-1) == original.size(-2)
    n_in = original.size(-1)
    center = slice((n_in + n%2 - n) // 2, (n_in + n%2 + n) // 2)
    return original[..., center, center]

class PhaseHologram(nn.Module):
    """
    This class calculates the far-field diffraction pattern of a phase hologram
    using the Fraunhofer diffraction equation.

    Parameters:
        phase: The phase of the diffractive optical element (DOE) that produces
               the hologram

    Subclassing nn.Module is not necessary, but is good practice because it
    provides assess to many useful class methods and functionality, particularly
    for complex models with many parameters and heirarchy.
    """
    def __init__(self, doe_diameter=2e-3, hologram_width=0.05,
                 wavelength=650e-9, device=None):
        """
        Args:
            wavelength: float, illumination wavelength, meters.
            doe_diameter: float, diameter of DOE, meters.
            hologram_width: float, angular extent of hologram, radians.
        """
        # Always call the initializer of the base class nn.Module.
        super().__init__()
        self.wavelength = wavelength

        # Calculate the sampling needed for the desired input/output sizes.
        self.num_elements = 2 * int(np.ceil(hologram_width * doe_diameter / wavelength))
        self.pitch = wavelength / (2 * hologram_width)
        self.angular_pitch = hologram_width / self.num_elements

        # Define the phase as a Parameter, which is a kind of tensor that has
        # special behavior when assigned as a Module attribute.
        #  * It is automatically added to the models parameter list.
        #  * It requires gradients by default.
        self.phase = nn.Parameter(torch.zeros((self.num_elements, self.num_elements)))

        # Define the input laser beam.
        # We'll use a Gaussian with a deviation equal to the radius of the DOE.
        x = self.pitch * (torch.arange(self.num_elements) - self.num_elements//2)

        r_squared = x**2 + x[:, None]**2  # Basically r^2 = x^2 + y^2

        sigma = doe_diameter / 2
        amp = torch.exp(-r_squared / (2*sigma**2))
        amp[r_squared > (doe_diameter**2)/4] = 0

        # Normalize the power in the input beam to 1
        amp /= (amp ** 2).sum().sqrt()

        self.register_buffer("amp", amp)

        self.to(device=device)

    def forward(self):
        field = self.amp * torch.exp(1j*self.phase)
        field = zeropad(field, 2*self.num_elements)
        angular_spectrum = fft.fftshift(fft.fft2(field, norm="ortho"))
        angular_spectrum = unpad(angular_spectrum, self.num_elements)
        return torch.abs(angular_spectrum)**2

    def plot(self):
        with torch.no_grad():
            phase = self.phase.detach().clone()
            phase = (phase + np.pi) % (2 * np.pi) - np.pi
            phase[self.amp == 0] = float('nan')

            I = self.forward().detach().cpu()

        fig, ax = plt.subplots(1, 2, figsize=(8,4))

        extent = [-self.num_elements/2 * self.pitch,
                  (self.num_elements/2 - 1) * self.pitch,
                  (self.num_elements/2 - 1) * self.pitch,
                  -self.num_elements/2 * self.pitch]
        ax[0].imshow(phase.cpu(), cmap='hsv', extent=extent)
        ax[0].set_title("DOE Phase")
        ax[0].axis('off')

        extent = [-self.num_elements/2 * self.angular_pitch,
                  (self.num_elements/2 - 1) * self.angular_pitch,
                  (self.num_elements/2 - 1) * self.angular_pitch,
                  -self.num_elements/2 * self.angular_pitch]
        ax[1].imshow(I, cmap='hot', extent=extent)
        ax[1].set_title("Diffraction Pattern")
        ax[1].axis('off')
        return fig, ax

model = PhaseHologram(device=device)


## 5. Forward Simulation

Replace the phase with a spiral pattern and visualize the result.

In [ ]:
n = model.num_elements
x = torch.arange(n) - n//2
x, y = torch.meshgrid(x, x, indexing='xy')
theta = torch.atan2(y, x)
with torch.no_grad():
    model.phase.copy_(theta)

f, ax = model.plot()

# This should produce a small "donut" beam. Zoom in so we can see it.
ax[1].axis((-0.001, 0.001, -0.001, 0.001))
plt.show()

## 6. Optimization Target

Upload a black-on-white line drawing.

Instructions:
1. Run the cell.
2. Upload a simple line-art image.
3. The image will be resized and inverted so the background is black and features are white.

In [ ]:
uploaded = files.upload()
name = list(uploaded.keys())[0] if uploaded else "goat.jpg"
img = Image.open(name).convert('L')
img = ImageOps.pad(img, (model.num_elements, model.num_elements), color=255)

target = torch.tensor(np.array(img) < 128, dtype=torch.float32)

plt.imshow(target, cmap='gray')
plt.title("Target")
plt.axis('off')
plt.show()


## 7. Merit Function

We combine efficiency and correlation into a single merit function.



In [ ]:
def efficiency(intensity, target):
    """ Calculates the fraction of the diffracted pattern that is on-target. """
    return (intensity*target).sum()

def correlation(intensity, target):
    """
    Calculates the normalized correlation coefficient between the
    diffraction pattern and the target
    """
    return (((intensity-intensity.mean())*(target-target.mean())).mean()
              /(intensity.std()*target.std()+1e-8))

## 8. Optimization

Train the hologram using gradient ascent on the merit function.

Experiment by changing the learning rate, optimizer, or merit weights.
Adjust `alpha` to trade off efficiency and image fidelity.

In [ ]:
learn_rate = 0.5
alpha = 0.5
max_iterations = 100

model = PhaseHologram().to(device)
target = target.to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=learn_rate, maximize=True)
eff_history = []
corr_history = []

for i in range(max_iterations):
    optimizer.zero_grad()
    I = model()
    eff = efficiency(I, target)
    corr = correlation(I, target)
    merit = alpha * eff + (1-alpha) * corr
    merit.backward()
    optimizer.step()
    eff_history.append(eff.item())
    corr_history.append(corr.item())

_ = model.plot()

f, ax = plt.subplots(1, 1, figsize=(4, 4))
ax.plot(eff_history, label='efficiency')
ax.plot(corr_history, label='correlation')
ax.set_ylim((0, 1))
ax.set_xlabel('Iteration')
ax.set_title("Merit")
ax.legend()
plt.show()